## Statistical Significance
- Evaluating the classifier models across different thresholds
- Each thresholds determines its performance and accuracy on properly labeling each text on diff. categories
- Testing each model's statistical significance using p-values and McNemar's Test

In [45]:
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parents[0]))
from src.tuning import show_metrics, visualize_metrics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# load the data
df = pd.read_csv('../data/processed/sms_cleaned_data.csv')
df

,date,text,carrier,target,punctuation_count,text_count,has_url,has_money_mention,has_phone_in_text,exclamation_count,uppercase_ratio,word_count,year,month
0,2026-07-01 09:13:32.668,login free spin marami nakakuha moneytoken sus...,Unknown,2,8,120,1.0,1.0,0.0,2.0,0.075000,9,2026,July
1,2026-04-23 01:01:22.884,condo across ateneo torre lorenzo studio units...,Smart,2,4,149,0.0,0.0,0.0,0.0,0.046980,15,2026,April
2,2026-03-27 06:23:44.260,papalduhin kita moneytoken ngayon basta bahala...,Smart,1,6,167,0.0,1.0,0.0,0.0,0.071856,22,2026,March
3,2026-02-17 10:25:35.838,boss panalunin kita pinata win moneytoken haba...,Smart,1,11,186,0.0,1.0,0.0,0.0,0.016129,20,2026,February
4,2026-01-19 16:13:33.766,nagsisimula saya panalo moneytoken free moneyt...,Smart,2,5,108,1.0,1.0,0.0,1.0,0.111111,11,2026,January
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1016,2018-02-28 10:37:26.541,shangri la updates risemakati one kind residen...,Smart,2,20,302,1.0,1.0,0.0,1.0,0.049669,29,2018,February
1017,2018-02-01 20:55:48.554,celebrate season love sweet staycation intimat...,Globe,2,5,160,0.0,0.0,0.0,0.0,0.037500,16,2018,February
1018,2018-01-24 12:44:09.178,new office space sale ortigas avenue near grov...,Smart,2,6,138,0.0,0.0,0.0,0.0,0.130435,17,2018,January
1019,2018-01-24 12:44:09.178,new office space sale ortigas avenue near grov...,Smart,2,6,138,0.0,0.0,0.0,0.0,0.130435,17,2018,January


## Tuned CNB and MNB Models

In [46]:
df['dedup_group_id'] = pd.factorize(df['text'])[0]

binary_cols = ['has_url', 'has_money_mention', 'has_phone_in_text']
numericals = ['exclamation_count', 'text_count', 'word_count',
              'punctuation_count', 'uppercase_ratio']

x = df['text']
y = df['target']

# splitting and vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.8, sublinear_tf=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scaler = MinMaxScaler(clip=True)
train_idx, test_idx = next(sgkf.split(x, y, groups=df['dedup_group_id']))

train_data = df.iloc[train_idx].reset_index(drop=True)
test_data = df.iloc[test_idx].reset_index(drop=True)

X_train = vectorizer.fit_transform(train_data['text'])
X_test = vectorizer.transform(test_data['text'])
y_train = train_data['target']
y_test = test_data['target']

num_train = scaler.fit_transform(train_data[numericals])
num_test  = scaler.transform(test_data[numericals])

bin_train = train_data[binary_cols].to_numpy(dtype=float)
bin_test  = test_data[binary_cols].to_numpy(dtype=float)

X_train = hstack([X_train, csr_matrix(bin_train), csr_matrix(num_train)]).tocsr()
X_test  = hstack([X_test,  csr_matrix(bin_test),  csr_matrix(num_test)]).tocsr()

# init both naive bayes model and using hyperparamters
mnb = MultinomialNB(alpha=0.1, fit_prior=False)
cnb = ComplementNB(alpha=0.1, norm=True)

mnb.fit(X_train, y_train)
cnb.fit(X_train, y_train)

ypred_mnb = mnb.predict(X_test)
ypred_cnb = cnb.predict(X_test)

yprob_mnb = mnb.predict_proba(X_test)
yprob_cnb = cnb.predict_proba(X_test)

print(f"Training Data Size: {len(train_data)} | Test Data Size: {len(test_data)}")

Training Data Size: 817 | Test Data Size: 204


## Tuned Random Forest Classifer

In [47]:
# 'rf__class_weight': None, 'rf__criterion': 'entropy', 'rf__max_depth': 100, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 4, 'rf__n_estimators': 200
rf = RandomForestClassifier(
    class_weight=None, criterion='entropy', max_depth=100,
    max_features='log2', min_samples_leaf=1, min_samples_split=4, n_estimators=200,
    random_state=42  
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)

show_metrics(y_test, rf_pred, rf_prob)

Accuracy and ROC AUC Score
Accuracy Score: 79.412%
ROC AUC Score: 90.264%
              precision    recall  f1-score   support

           0       0.75      0.74      0.75        54
           1       0.78      0.88      0.83        82
           2       0.85      0.74      0.79        68

    accuracy                           0.79       204
   macro avg       0.79      0.78      0.79       204
weighted avg       0.80      0.79      0.79       204



### Helper Function for Defining Thresholds

In [48]:
unique_labels = np.unique(y_test)
def set_threshold(yprob, ytest, confidence, model, model_name, labels=unique_labels):

    max_probs = np.max(yprob, axis=1)
    best_classes = np.argmax(yprob, axis=1)
    threshold = np.where(max_probs >= confidence, best_classes, -1)

    accuracy = accuracy_score(y_test, threshold)

    print(f"===Metrics within {confidence * 100}% Threshold===")
    print(f"Accuracy Score: {(accuracy * 100):.3f}%")
    print(classification_report(ytest, threshold, labels=labels, zero_division=0))

    cm = confusion_matrix(ytest, threshold, labels=labels)
    visualize_metrics(
        model=model,
        cm=cm,
        xtest=X_test,
        ytest=ytest,
        name=f"{model_name} (>{confidence*100:.0f}%)",
        labels=['Smishing', 'Legitimate', 'Spam']
    )

In [49]:
# metrics without threshold tuning
show_metrics(y_test, ypred_cnb, yprob_cnb)

Accuracy and ROC AUC Score
Accuracy Score: 74.020%
ROC AUC Score: 89.096%
              precision    recall  f1-score   support

           0       0.74      0.69      0.71        54
           1       0.75      0.78      0.77        82
           2       0.72      0.74      0.73        68

    accuracy                           0.74       204
   macro avg       0.74      0.73      0.74       204
weighted avg       0.74      0.74      0.74       204



In [50]:
show_metrics(y_test, ypred_mnb, yprob_mnb)

Accuracy and ROC AUC Score
Accuracy Score: 73.529%
ROC AUC Score: 88.484%
              precision    recall  f1-score   support

           0       0.77      0.67      0.71        54
           1       0.76      0.78      0.77        82
           2       0.68      0.74      0.71        68

    accuracy                           0.74       204
   macro avg       0.74      0.73      0.73       204
weighted avg       0.74      0.74      0.74       204

